#task 1

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate


model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


template = """
You are a helpful AI assistant.
Answer clearly:

Question: {question}
"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)


def generate_response(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


questions = [
    "What is Machine Learning?",
    "Explain Python in simple terms",
    "What is cloud computing?"
]

for q in questions:
    formatted_prompt = prompt.format(question=q)
    response = generate_response(formatted_prompt)

    print("\nQUESTION:", q)
    print("ANSWER:", response)

c:\Gen ai\venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Gen ai\venv\lib\site-packages\transformers\generation\utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(



QUESTION: What is Machine Learning?
ANSWER: Machine learning is a technology that enables people to learn from their past.

QUESTION: Explain Python in simple terms
ANSWER: Python is a programming language that uses a set of digits to represent a

QUESTION: What is cloud computing?
ANSWER: a computer program


#task 2

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful AI tutor."),
    HumanMessage(content="What is Python?"),
    AIMessage(content="Python is a programming language."),  
    HumanMessage(content="Explain its uses in detail.")
])


messages = chat_prompt.format_messages()

final_prompt = "\n".join([msg.content for msg in messages])


def generate_response(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

response = generate_response(final_prompt)

print("FINAL PROMPT:\n", final_prompt)
print("\nRESPONSE:\n", response)

FINAL PROMPT:
 You are a helpful AI tutor.
What is Python?
Python is a programming language.
Explain its uses in detail.

RESPONSE:
 Python is a programming language that is used to code programs.


#task 3

In [6]:

from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM



class Answer(BaseModel):
    answer: str = Field(description="Final answer to the user question")
    confidence: float = Field(description="Confidence score between 0 and 1")
    source: str = Field(description="Source of the answer")



parser = PydanticOutputParser(pydantic_object=Answer)
format_instructions = parser.get_format_instructions()

print("FORMAT INSTRUCTIONS:\n", format_instructions)


template = """
You are a helpful assistant.

You MUST return output in STRICT JSON format only.

{format_instructions}

Rules:
- Output must be valid JSON
- Do not add extra text
- Do not explain anything

Question: {question}
"""

prompt = PromptTemplate(
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions},
    template=template
)



model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def generate_response(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)



def safe_generate_and_parse(prompt_text, retries=3):
    for i in range(retries):
        raw_output = generate_response(prompt_text)
        print(f"\n🔹 Attempt {i+1} RAW OUTPUT:\n{raw_output}")

        try:
            parsed = parser.parse(raw_output)
            return parsed
        except Exception as e:
            print("⚠️ Parsing failed:", e)

    return None



question = "What is Machine Learning?"

formatted_prompt = prompt.format(question=question)

result = safe_generate_and_parse(formatted_prompt)



if result:
    print("\n STRUCTURED OUTPUT:")
    print(result)
    print("\nFields:")
    print("Answer:", result.answer)
    print("Confidence:", result.confidence)
    print("Source:", result.source)
else:
    print("\n Failed after retries. Using fallback.")

    fallback = Answer(
        answer="Unable to generate structured response",
        confidence=0.0,
        source="fallback"
    )

    print(fallback)

FORMAT INSTRUCTIONS:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"description": "Final answer to the user question", "title": "Answer", "type": "string"}, "confidence": {"description": "Confidence score between 0 and 1", "title": "Confidence", "type": "number"}, "source": {"description": "Source of the answer", "title": "Source", "type": "string"}}, "required": ["answer", "confidence", "source"]}
```

🔹 Attempt 1 RAW OUTPUT:
Machine learning is a type of machine learning that uses machine learning to learn information about people.
⚠️ Parsing failed: Invalid 

#task 4

In [7]:
def safe_parse(prompt_text, max_retries=3):
    for attempt in range(max_retries):
        raw_output = generate_response(prompt_text)
        
        try:
            parsed = parser.parse(raw_output)
            return parsed
        
        except Exception as e:
            print(f"Attempt {attempt+1} failed:", e)
    
    return None
result = safe_parse(formatted_prompt)

if result:
    print("\n Valid Output:")
    print(result)
else:
    print("\n Failed to parse. Using fallback.")

    fallback = Answer(
        answer="Sorry, I couldn't generate a structured response.",
        confidence=0.0,
        source="fallback"
    )
    
    print(fallback)

Attempt 1 failed: Invalid json output: Machine learning is a type of machine learning that uses machine learning to learn information about people.
Attempt 2 failed: Invalid json output: Machine learning is a type of machine learning that uses machine learning to learn information about people.
Attempt 3 failed: Invalid json output: Machine learning is a type of machine learning that uses machine learning to learn information about people.

 Failed to parse. Using fallback.
answer="Sorry, I couldn't generate a structured response." confidence=0.0 source='fallback'


#task 5,6,7

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_text(prompt):
    prompt = prompt.to_string() if hasattr(prompt, "to_string") else str(prompt)

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=100)
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


model_runnable = RunnableLambda(generate_text)


simple_prompt = PromptTemplate.from_template(
    "Answer clearly:\n{question}"
)

chain = simple_prompt | model_runnable | StrOutputParser()

print(" Simple Chain Output:")
print(chain.invoke({"question": "What is AI?"}))




def retriever_func(input_dict):
    q = input_dict["question"]
    return f"[Retrieved Info] {q} is a factual query with stored knowledge."

retriever = RunnableLambda(retriever_func)

llm_prompt = PromptTemplate.from_template(
    "Answer normally:\n{question}"
)

llm_chain = llm_prompt | model_runnable | StrOutputParser()

def route(input_dict):
    question = input_dict["question"].lower()
    
    if "what" in question or "who" in question or "when" in question:
        return retriever.invoke(input_dict)
    else:
        return llm_chain.invoke(input_dict)

conditional_chain = RunnableLambda(route)

print("\n Conditional Chain Output:")
print(conditional_chain.invoke({"question": "What is Machine Learning?"}))
print(conditional_chain.invoke({"question": "Tell me a joke"}))




answer_prompt = PromptTemplate.from_template(
    "Answer:\n{question}"
)

summary_prompt = PromptTemplate.from_template(
    "Summarize in one line:\n{question}"
)

answer_chain = answer_prompt | model_runnable | StrOutputParser()
summary_chain = summary_prompt | model_runnable | StrOutputParser()

parallel_chain = RunnableParallel(
    answer=answer_chain,
    summary=summary_chain
)

print("\n Parallel Chain Output:")
result = parallel_chain.invoke({"question": "Explain Artificial Intelligence in detail"})

print("Answer:\n", result["answer"])
print("Summary:\n", result["summary"])

c:\Gen ai\venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


 Simple Chain Output:
Artificial intelligence

 Conditional Chain Output:
[Retrieved Info] What is Machine Learning? is a factual query with stored knowledge.
a snooty snooty

 Parallel Chain Output:
Answer:
 What is the main idea of this article?
Summary:
 Understand the concept of artificial intelligence.


#task 8

In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel


model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def generate_text(prompt):
    prompt = prompt.to_string() if hasattr(prompt, "to_string") else str(prompt)

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=100)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

llm = RunnableLambda(generate_text)   
prompt = PromptTemplate.from_template("Tell me a joke about {topic}")

chain = prompt | llm | StrOutputParser()

print(" Joke:")
print(chain.invoke({"topic": "AI"}))


joke_prompt = PromptTemplate.from_template("Tell me a joke about {topic}")
explain_prompt = PromptTemplate.from_template("Explain this joke: {joke}")


joke_chain = (
    {"topic": RunnablePassthrough()}
    | joke_prompt
    | llm
    | StrOutputParser()
)


final_chain = RunnableParallel({
    "joke": joke_chain,
    "explanation": joke_chain | explain_prompt | llm | StrOutputParser()
})

result = final_chain.invoke("programming")

print("\n Parallel Output:")
print(result)


chain = (
    PromptTemplate.from_template("Explain {topic} in simple terms")
    | llm
    | StrOutputParser()
)

print("\n Explanation:")
print(chain.invoke({"topic": "Machine Learning"}))

c:\Gen ai\venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


 Joke:
i'm a robot

 Parallel Output:
{'joke': 'a program that is a computer program is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is', 'explanation': 'a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program that is a computer program'}

 Explanation:
Machine Learning is a

#task 9

In [21]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def retriever_func(question):
    return [
        type("Doc", (), {"page_content": f"Info about {question} part 1"}),
        type("Doc", (), {"page_content": f"Info about {question} part 2"}),
    ]

retriever = RunnableLambda(retriever_func)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

format_runnable = RunnableLambda(format_docs)


prompt_template = PromptTemplate.from_template(
    "Answer the question based on context:\n\n{context}\n\nQuestion: {question}"
)
prompt_runnable = RunnableLambda(lambda d: prompt_template.format(**d))


rag_chain = (
    RunnableParallel({
        "context": retriever | format_runnable,
        "question": RunnablePassthrough()
    })
    | prompt_runnable
    | llm
    | StrOutputParser()
)

queries = [
    "What is AI?",
    "Explain embeddings",
    "What is RAG?"
]

for q in queries:
    print("Q:", q)
    print("A:", rag_chain.invoke({"question": q}))
    print("-" * 50)

Q: What is AI?
A: part 1
--------------------------------------------------
Q: Explain embeddings
A: part 1
--------------------------------------------------
Q: What is RAG?
A: part 1
--------------------------------------------------


#task 10

# Why structured output is important

- Ensures consistent and predictable responses (e.g., JSON format)
- Makes it easier to integrate with APIs and frontends
- Reduces ambiguity in LLM outputs
- Helps in automation and downstream processing

---

## Advantages of LCEL over traditional chains

- More flexible and modular pipeline design
- Supports parallel and conditional execution
- Cleaner and more readable syntax using |
- Easier debugging and testing
- Better composability with reusable components

---

##  When to use parallel vs conditional chains

### Parallel Chains
- Use when tasks are independent
- Improves performance by running tasks simultaneously
- Example: generate answer, summary, and keywords at the same time

### Conditional Chains
- Use when execution depends on input type or logic
- Example:
  - If question is factual → use retriever (RAG)
  - Else → use direct LLM response